In [1]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

## Load dataset generated by CovaS

In [2]:
def calculate_macro_tpr_fpr(voting_cm):
    num_classes = voting_cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        TP = voting_cm[i, i]
        FN = np.sum(voting_cm[i, :]) - TP
        FP = np.sum(voting_cm[:, i]) - TP
        TN = np.sum(voting_cm) - (TP + FN + FP)

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

        tpr_list.append(TPR)
        fpr_list.append(FPR)

    macro_tpr = np.mean(tpr_list)
    macro_fpr = np.mean(fpr_list)

    return macro_tpr, macro_fpr

train = pd.read_csv('/dis/DS/hungnt/CICModbus2023/train_70_600.csv')
test = pd.read_csv('/dis/DS/hungnt/CICModbus2023/test_30_600.csv')

In [6]:
# Lấy indices của từng nhãn cần giảm mẫu
indices_to_keep = []

# Với nhãn 0 và 2: random sample 14000 mẫu
for label in [0, 2]:
    label_indices = train[train['Label'] == label].index
    sampled_indices = np.random.choice(label_indices, size=1400, replace=False)
    indices_to_keep.extend(sampled_indices)

# Giữ nguyên tất cả mẫu của các nhãn khác
other_labels = [1, 3, 4, 5, 6, 7, 8]
for label in other_labels:
    label_indices = train[train['Label'] == label].index
    indices_to_keep.extend(label_indices)

# Tạo dataframe mới với mẫu đã chọn
train = train.loc[indices_to_keep].reset_index(drop=True)

print("Class distribution sau khi sampling:")
print(train['Label'].value_counts().sort_index())
print(f"\nTotal samples: {len(train)}")

Class distribution sau khi sampling:
Label
0    1400
1    1285
2    1400
3      60
4    1064
5     983
6    1043
7    1058
8     913
Name: count, dtype: int64

Total samples: 9206


In [7]:
train.to_csv('/dis/DS/hungnt/CICModbus2023/train_70_balanced_600.csv', index=False)

# AutoGluon Tabular

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.metrics import classification_report

label = 'Label'

predictor = TabularPredictor(
    label=label,
    problem_type='multiclass',
    eval_metric='f1_macro',
    path='AG_CICModbus_FULL'
).fit(
    train_data=train,
    presets='best_quality',
    time_limit=12*3600,
    num_bag_folds=5,
    num_bag_sets=2,
    num_stack_levels=2,
    num_gpus=1,
    verbosity=2
)

print(predictor.leaderboard(test, silent=True))

y_true = test[label]
y_pred = predictor.predict(test)

print(classification_report(y_true, y_pred, digits=4))

print(predictor.get_model_best())


# Training model

In [8]:
X_train = train.drop(['Label'], axis=1)
y_train = train['Label']
X_test = test.drop(['Label'], axis=1)
y_test = test['Label']

In [9]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

## XGBoost

In [17]:
xgb_params = {
    'tree_method': 'gpu_hist',
    'predictor': 'gpu_predictor',
    'max_depth': 8,
    'n_estimators': 3000,
    'learning_rate': 0.1,
    'eval_metric': 'mlogloss',
    'objective': 'multi:softprob',
    'num_class': len(np.unique(y_train)),
    'booster': 'gbtree',
    'random_state': 42,
    'early_stopping_rounds': 50,
    'n_jobs': -1
}

train_weight = compute_sample_weight("balanced", y_train_split)
val_weight   = compute_sample_weight("balanced", y_val_split)

print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)

xgb_model.fit(
    X_train_split, y_train_split,
    sample_weight=train_weight,
    eval_set=[(X_val_split, y_val_split)],
    sample_weight_eval_set=[val_weight],
    verbose=False
)
print("XGBClassifier Finished")

xgb_start_time = time.time()
xgb_prediction = xgb_model.predict(X_test)
xgb_end_time = time.time()
xgb_time = xgb_end_time - xgb_start_time
xgb_acc = sklearn.metrics.accuracy_score(y_test, xgb_prediction)
xgb_precision = sklearn.metrics.precision_score(y_test, xgb_prediction, average='macro')
xgb_f1 = sklearn.metrics.f1_score(y_test, xgb_prediction, average='macro')
xgb_recall = sklearn.metrics.recall_score(y_test, xgb_prediction, average='macro')
xgb_cm = sklearn.metrics.confusion_matrix(y_test, xgb_prediction)

print("XGBoost report:")
print("XGBoost Time:", xgb_time)
print("XGBoost Accuracy:", xgb_acc)
print("XGBoost Precision:", xgb_precision)
print("XGBoost F1:", xgb_f1)
print("XGBoost Recall:", xgb_recall)
print("XGBoost CM:\n", xgb_cm)
xgb_tpr, xgb_fpr = calculate_macro_tpr_fpr(xgb_cm)
print(f'XGBoost Macro-average TPR: {xgb_tpr}')
print(f'XGBoost Macro-average FPR: {xgb_fpr}')
print(classification_report(y_test, xgb_prediction, digits=4))

XGBClassifier Starting
XGBClassifier Finished
XGBoost report:
XGBoost Time: 0.03369641304016113
XGBoost Accuracy: 0.9401926001013685
XGBoost Precision: 0.9442927009743484
XGBoost F1: 0.9448274179003346
XGBoost Recall: 0.9457504679432992
XGBoost CM:
 [[594   2   0   0   0   1   2   1   0]
 [  1 498   2   0   0  15   2   0  33]
 [  0  11 536   0   0   7   0   0  46]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  1  17   0   0   0 386  10   0   7]
 [  0  13   0   0   0   3 430   0   1]
 [  1   0   0   0   2   0   0 450   0]
 [  1  13  32   0   1  10   0   0 335]]
XGBoost Macro-average TPR: 0.9457504679432992
XGBoost Macro-average FPR: 0.007552971101501242
              precision    recall  f1-score   support

           0     0.9933    0.9900    0.9917       600
           1     0.8989    0.9038    0.9014       551
           2     0.9404    0.8933    0.9162       600
           3     1.0000    1.0000    1.0000        26
           4     0.9934    0.9978

## ExtraTree

In [11]:
et_params = {
    "n_estimators": 100,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
et_prediction = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")

et_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=et_prediction)
et_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=et_prediction)
et_fp = et_cm[0, 1]
print("ExtraTrees report:")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees CM:\n", et_cm)
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')
print(classification_report(y_test, et_prediction, digits=4))

ExtraTreesClassifier Starting


ExtraTreesClassifier Finished
ExtraTrees report:
ExtraTrees Time: 0.17228007316589355
ExtraTrees Accuracy: 0.9490623416117587
ExtraTrees Precision: 0.9537264351653957
ExtraTrees F1: 0.9541319829405889
ExtraTrees Recall: 0.9558127568808119
ExtraTrees CM:
 [[594   3   0   0   0   2   0   1   0]
 [  7 523   4   0   0   9   0   0   8]
 [  0  17 511   0   0  19   0   0  53]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  9  10   0   0   0 400   1   0   1]
 [  5   4   0   0   0   6 431   0   1]
 [  2   0   0   0   2   0   0 449   0]
 [  1  12  15   0   0   7   1   0 356]]
ExtraTrees Macro-average TPR: 0.9558127568808119
ExtraTrees Macro-average FPR: 0.006449147813887304
              precision    recall  f1-score   support

           0     0.9612    0.9900    0.9754       600
           1     0.9192    0.9492    0.9339       551
           2     0.9642    0.8517    0.9044       600
           3     1.0000    1.0000    1.0000        26
           4     0.99

## RandomForest

In [12]:
rf_params = {
    "n_estimators": 100,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy",
    "class_weight": "balanced"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
rf_prediction = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")

rf_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=rf_prediction)
rf_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=rf_prediction)
rf_fp = rf_cm[0, 1]
print("RandomForest report:")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest CM:\n", rf_cm)
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')
print(classification_report(y_test, rf_prediction, digits=4))

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest report:
RandomForest Time: 0.18923187255859375
RandomForest Accuracy: 0.9564115560060821
RandomForest Precision: 0.9603111891619189
RandomForest F1: 0.9604510529064042
RandomForest Recall: 0.9616175263170779
RandomForest CM:
 [[596   3   0   0   1   0   0   0   0]
 [  0 530   1   0   0   6   0   0  14]
 [  0  15 527   0   0   9   0   0  49]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 455   0   1   0   0]
 [  4  13   0   0   0 402   1   0   1]
 [  2  10   0   0   0   2 432   0   1]
 [  2   0   0   0   2   0   0 449   0]
 [  0  13  14   0   0   7   1   0 357]]
RandomForest Macro-average TPR: 0.9616175263170779
RandomForest Macro-average FPR: 0.005510485762713642
              precision    recall  f1-score   support

           0     0.9868    0.9933    0.9900       600
           1     0.9075    0.9619    0.9339       551
           2     0.9723    0.8783    0.9229       600
           3     1.0000 

## Lightgbm

In [13]:
import time
import lightgbm as lgb

lgbm_params = {
    'boosting_type': 'gbdt',
    'objective': 'multiclass',
    'num_class': int(y_train_split.nunique()),
    'learning_rate': 0.1,
    'max_depth': 8,
    'n_estimators': 3000,
    'device_type': 'cpu',
    'metric': ['multi_logloss', 'multi_error'],
    'random_state': 42,
    'class_weight': 'balanced'
}

print("LGBMClassifier Starting")
lgbm_model = lgb.LGBMClassifier(**lgbm_params)

lgbm_model.fit(
    X_train_split, y_train_split,
    eval_set=[(X_val_split, y_val_split)],
    eval_metric=['multi_logloss', 'multi_error'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
    verbose=False
)

lgbm_start_time = time.time()
lgbm_prediction = lgbm_model.predict(X_test)
lgbm_end_time = time.time()
lgbm_time = lgbm_end_time - lgbm_start_time
print("LGBMClassifier Finished")

lgbm_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=lgbm_prediction)
lgbm_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=lgbm_prediction, average='macro')
lgbm_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=lgbm_prediction)

print("LightGBM report:")
print("LightGBM Time:", lgbm_time)
print("LightGBM Accuracy:", lgbm_acc)
print("LightGBM Precision:", lgbm_precision)
print("LightGBM F1:", lgbm_f1)
print("LightGBM Recall:", lgbm_recall)
print("LightGBM CM:\n", lgbm_cm)
lgbm_tpr, lgbm_fpr = calculate_macro_tpr_fpr(lgbm_cm)
print(f'LightGBM Macro-average TPR: {lgbm_tpr}')
print(f'LightGBM Macro-average FPR: {lgbm_fpr}')
print(classification_report(y_test, lgbm_prediction, digits=4))

LGBMClassifier Starting


/home/soc/miniconda3/envs/soc/lib/python3.8/site-packages/lightgbm/sklearn.py:736: UserWarning: 'verbose' argument is deprecated and will be removed in a future release of LightGBM. Pass 'log_evaluation()' callback via 'callbacks' argument instead.
  _log_warning("'verbose' argument is deprecated and will be removed in a future release of LightGBM. "


LGBMClassifier Finished
LightGBM report:
LightGBM Time: 0.07085871696472168
LightGBM Accuracy: 0.9381652306132793
LightGBM Precision: 0.9429073767421685
LightGBM F1: 0.9432294067052359
LightGBM Recall: 0.9440267679826745
LightGBM CM:
 [[591   3   1   0   1   1   2   1   0]
 [  2 502   3   0   0  11   3   0  30]
 [  0  12 531   0   0   7   0   0  50]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 454   0   1   0   1]
 [  0  20   0   0   0 391   5   0   5]
 [  0  13   0   0   0   7 427   0   0]
 [  1   1   0   0   2   0   0 449   0]
 [  1  19  28   0   1  11   1   0 331]]
LightGBM Macro-average TPR: 0.9440267679826745
LightGBM Macro-average FPR: 0.007816120107912647
              precision    recall  f1-score   support

           0     0.9933    0.9850    0.9891       600
           1     0.8807    0.9111    0.8956       551
           2     0.9432    0.8850    0.9132       600
           3     1.0000    1.0000    1.0000        26
           4     0.9913    0.9956    0.9934   

## catboost

In [15]:
import time
from catboost import CatBoostClassifier

cat_params = {
    'task_type': 'GPU',
    'iterations': 3000,
    'depth': 8,
    'learning_rate': 0.1,
    'loss_function': 'MultiClass',
    'eval_metric': 'MultiClass',
    'random_seed': 42,
    'od_type': 'Iter', 
    'od_wait': 50,
    'verbose': False,
    'auto_class_weights': 'Balanced'
}

print("CatBoostClassifier Starting")
cat_model = CatBoostClassifier(**cat_params)
cat_model.fit(
    X_train_split, y_train_split,
    eval_set=(X_val_split, y_val_split),
    use_best_model=True,
    verbose=False
)

cat_start_time = time.time()
cat_prediction = cat_model.predict(X_test).astype(int).ravel()
cat_end_time = time.time()
cat_time = cat_end_time - cat_start_time
print("CatBoostClassifier Finished")

cat_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=cat_prediction)
cat_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=cat_prediction, average='macro')
cat_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=cat_prediction)

print("CatBoost report:")
print("CatBoost Time:", cat_time)
print("CatBoost Accuracy:", cat_acc)
print("CatBoost Precision:", cat_precision)
print("CatBoost F1:", cat_f1)
print("CatBoost Recall:", cat_recall)
print("CatBoost CM:\n", cat_cm)
cat_tpr, cat_fpr = calculate_macro_tpr_fpr(cat_cm)
print(f'CatBoost Macro-average TPR: {cat_tpr}')
print(f'CatBoost Macro-average FPR: {cat_fpr}')
print(classification_report(y_test, cat_prediction, digits=4))

CatBoostClassifier Starting
CatBoostClassifier Finished
CatBoost report:
CatBoost Time: 0.03521227836608887
CatBoost Accuracy: 0.9272681196147998
CatBoost Precision: 0.9326168713770056
CatBoost F1: 0.9330931243674156
CatBoost Recall: 0.9344675258788757
CatBoost CM:
 [[593   3   0   0   1   1   2   0   0]
 [  3 470   2   0   0  15   4   0  57]
 [  0  12 530   0   0   8   0   0  50]
 [  0   0   0  26   0   0   0   0   0]
 [  0   0   0   0 452   0   1   0   3]
 [  1  16   1   0   0 388  10   0   5]
 [  1   8   0   0   0   8 430   0   0]
 [  2   1   0   0   2   0   0 447   1]
 [  2  27  26   0   0  14   0   0 323]]
CatBoost Macro-average TPR: 0.9344675258788757
CatBoost Macro-average FPR: 0.009166533689052769
              precision    recall  f1-score   support

           0     0.9850    0.9883    0.9867       600
           1     0.8752    0.8530    0.8640       551
           2     0.9481    0.8833    0.9146       600
           3     1.0000    1.0000    1.0000        26
           4  